# Namma Space v2: frames -> COLMAP -> Gaussian splat
Set **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. Approve the Drive pop-up when it appears.
Fixes vs v1: drops blurry frames, uses exhaustive matching, **picks the largest COLMAP model** (v1 picked a 4-image fragment), small Drive backup, and saves the training log so errors are visible.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, os, shutil, json, time, subprocess, zipfile
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)

ROOM="room1"; ZIP_NAME="room1_images.zip"
MAX_ITERS=15000          # use 30000 for the final run
BLUR_MIN=60              # frames sharper than this are kept
STEP=2                   # use every 2nd frame
WORK="/content/work"; IMAGES=f"{WORK}/images"; SUB=f"{WORK}/images_sub"
NS=f"{WORK}/ns"; OUT=f"{WORK}/outputs"; EXPORT=f"{WORK}/export"

hits=glob.glob(f"/content/drive/MyDrive/Clg*/namma*/{ZIP_NAME}")
assert hits, "Zip not found in Drive"
ZIP_PATH=hits[0]; DRIVE_DIR=os.path.dirname(ZIP_PATH)
BACKUP=f"{DRIVE_DIR}/{ROOM}_backup"; os.makedirs(BACKUP,exist_ok=True)
print("Zip:",ZIP_PATH)

## 2. Install (5 to 10 min; red pip warnings are fine)

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y colmap ffmpeg ninja-build > /dev/null
pip -q install nerfstudio ninja
colmap -h | head -n 2

## 3. Unzip, drop blurry frames, keep every 2nd

In [ ]:
import cv2
shutil.rmtree(IMAGES,ignore_errors=True); os.makedirs(IMAGES)
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(IMAGES)
nested=os.path.join(IMAGES,"images")
if os.path.isdir(nested):
    for f in os.listdir(nested): shutil.move(os.path.join(nested,f),IMAGES)
    os.rmdir(nested)
fr=sorted(glob.glob(f"{IMAGES}/*.jp*g")+glob.glob(f"{IMAGES}/*.png"))
sc=[cv2.Laplacian(cv2.imread(f,0),cv2.CV_64F).var() for f in fr]
sharp=[f for f,s in zip(fr,sc) if s>=BLUR_MIN]
shutil.rmtree(SUB,ignore_errors=True); os.makedirs(SUB)
for f in sharp[::STEP]: shutil.copy(f,SUB)
print(len(fr),"frames,",len(fr)-len(sharp),"blurry dropped,",len(os.listdir(SUB)),"sent to COLMAP")

## 4. COLMAP poses (restores from Drive if a good backup exists)

In [ ]:
def n_frames(p):
    try: return len(json.load(open(p))["frames"])
    except Exception: return 0

if n_frames(f"{BACKUP}/ns/transforms.json")>=40 and not os.path.exists(f"{NS}/transforms.json"):
    print("Restoring good COLMAP result from Drive..."); shutil.copytree(f"{BACKUP}/ns",NS)
elif not os.path.exists(f"{NS}/transforms.json"):
    t=time.time()
    !ns-process-data images --data {SUB} --output-dir {NS} --matching-method exhaustive --no-gpu
    print(f"COLMAP took {(time.time()-t)/60:.1f} min")

## 5. Use the LARGEST COLMAP model, rebuild transforms.json, check

In [ ]:
import subprocess, re
from pathlib import Path
from nerfstudio.process_data.colmap_utils import colmap_to_json
S=f"{NS}/colmap/sparse"
def reg(m):
    o=subprocess.run(["colmap","model_analyzer","--path",f"{S}/{m}"],capture_output=True,text=True)
    r=re.search(r"Registered images: (\d+)",o.stdout+o.stderr); return int(r.group(1)) if r else 0
models=sorted(os.listdir(S)); sizes={m:reg(m) for m in models}; print("models:",sizes)
best=max(sizes,key=sizes.get)
if best!="0":
    shutil.move(f"{S}/0",f"{S}/tiny_old"); shutil.move(f"{S}/{best}",f"{S}/0")
n=colmap_to_json(recon_dir=Path(S)/"0",output_dir=Path(NS))
total=len(os.listdir(SUB)); print(f"{n} of {total} frames registered ({100*n/total:.0f}%)")
assert n>=40, "Too few poses: re-film slower with more overlap before training."
# small backup (skips the resized image copies; avoids Drive quota problems)
try:
    shutil.rmtree(f"{BACKUP}/ns",ignore_errors=True); os.makedirs(f"{BACKUP}/ns")
    for it in ["transforms.json","sparse_pc.ply","colmap","images"]:
        s=f"{NS}/{it}"
        if os.path.isdir(s): shutil.copytree(s,f"{BACKUP}/ns/{it}")
        elif os.path.exists(s): shutil.copy(s,f"{BACKUP}/ns/")
    print("Backed up to Drive.")
except Exception as e: print("Backup skipped (Drive full?):",e)

## 6. Train (20 to 40 min). The log is saved so any error is visible

In [ ]:
shutil.rmtree(OUT,ignore_errors=True)
t=time.time()
cmd=f"ns-train splatfacto --data {NS} --output-dir {OUT} --experiment-name {ROOM} --max-num-iterations {MAX_ITERS} --vis tensorboard --viewer.quit-on-train-completion True"
rc=subprocess.run(cmd+" > /content/train.log 2>&1",shell=True).returncode
print(f"exit code {rc}, took {(time.time()-t)/60:.1f} min")
if rc!=0 or not glob.glob(f"{OUT}/**/*.ckpt",recursive=True):
    print("TRAINING FAILED. Last 40 lines of the log:\n")
    print(subprocess.run("tail -40 /content/train.log",shell=True,capture_output=True,text=True).stdout)
    raise SystemExit("Send the lines above to your helper.")

## 7. Export

In [ ]:
cfg=sorted(glob.glob(f"{OUT}/**/config.yml",recursive=True))[-1]
shutil.rmtree(EXPORT,ignore_errors=True)
!ns-export gaussian-splat --load-config {cfg} --output-dir {EXPORT}
plys=glob.glob(f"{EXPORT}/*.ply"); assert plys,"export failed"
PLY=plys[0]; print(PLY,round(os.path.getsize(PLY)/1e6,1),"MB")

## 8. Save to Drive, and download directly as a fallback

In [ ]:
try:
    shutil.copy(PLY,f"{DRIVE_DIR}/{ROOM}.ply"); print("Saved to Drive:",f"{DRIVE_DIR}/{ROOM}.ply")
except Exception as e:
    print("Drive save failed (quota?):",e)
from google.colab import files
files.download(PLY)